**Week 1:  Import raw data, build variable master list, map survey responses to analysis categories using Appendix A.**
    
Step 1: Import raw data

In [ ]:
# Libraries importation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Uploading the dataset

dataset = pd.ExcelFile("2024 ALS Totals for SWB.xlsx")

In [ ]:
# Dataset Tables

print(dataset.sheet_names)

**Exploring Table Dataset**

In [ ]:
table_df = pd.read_excel("2024 ALS Totals for SWB.xlsx", sheet_name = "Table")

In [ ]:
table_df.info()

In [ ]:
table_df.describe(include='all')

In [ ]:
table_df.columns

In [ ]:
table_df.select_dtypes("float").nunique()

In [ ]:
table_df.select_dtypes("int").nunique()

In [ ]:
table_df.select_dtypes("object").nunique()

In [ ]:
# Checking for null values

null_counts = table_df.isnull().sum()
print(null_counts)

In [ ]:
# Visualizing Null values

plt.figure(figsize=(12, 6))
sns.heatmap(table_df.isnull(), cmap='viridis')
plt.title("Missing Data Heatmap")
plt.show()

In [ ]:
print(table_df.columns.tolist())

In [ ]:
# count completely empty columns

empty_cols = table_df.columns[table_df.isnull().all()]
print(f"Number of completely empty columns: {len(empty_cols)}")
print("Empty column names:", list(empty_cols))

**Data Cleaning**

In [ ]:
# Drop Empty Columns

table_df = table_df.dropna(axis=1, how='all')

In [ ]:
# Standardize Column Names

table_df.columns = (
    table_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace('\xa0', '_')
    .str.replace(r'[^\w\s]', '', regex=True)
)
table_df.columns

In [ ]:
# Identify fully empty rows

empty_rows = table_df[table_df.isnull().all(axis=1)]

# Count them
print(f"Number of fully empty rows: {len(empty_rows)}")
display(empty_rows)

In [ ]:
#Count duplicate rows

duplicate_count = table_df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

In [ ]:
#View duplicate rows

duplicates = table_df[table_df.duplicated(keep=False)]
display(duplicates)

In [ ]:
table_df = table_df.drop_duplicates()

In [ ]:
print(table_df.info())

In [ ]:
# Documenting Missing Data

missing_data = table_df.isnull().sum().reset_index()
missing_data.columns = ['Column_name', 'Missing_Values']
missing_data['Missing_Percentage'] = ((missing_data['Missing_Values'] / len(table_df)) * 100).round(2)
missing_data['DataType'] = table_df.dtypes.values

missing_data = missing_data[missing_data['Missing_Values'] > 0]

missing_data.to_csv('missing_values_report.csv', index=False)

In [ ]:
# Dropping row with no league id

missing_row = table_df[table_df["league_id"].isnull()]

missing_row = missing_row.dropna(subset = ["league_id"])

**Mapping Responses**

In [ ]:
# Mapping columns: 45-66...based on the analysis plan, we have only these columns within the range: 56, 58, and 62.
# 56

revenue_estimates_column = [
    'dues_56_please_indicate_which_and_offer_estimates_of_your_revenue_streams_for_2024',
    'grants_from_lwvus_56_please_indicate_which_and_offer_estimates_of_your_revenue_streams_for_2024',
    'external_grants_56_please_indicate_which_and_offer_estimates_of_your_revenue_streams_for_2024',
    'fundraising_efforts_56_please_indicate_which_and_offer_estimates_of_your_revenue_streams_for_2024',
    'individual_donations_from_your_league_members_56_please_indicate_which_and_offer_estimates_of_your_revenue_streams_for_2024',
    'in_kind_donation_not_including_volunteer_time_56_please_indicate_which_and_offer_estimates_of_your_revenue_streams_for_2024'
]

mappings = {
    56: {
        "Dues": {
            '$500 or less': 'revenue estimates is less or equal to $500',
            '$501-$1,500': 'revenue estimates is between $501 to $1500',
            '$1,501-$3,000': 'revenue estimates is between $1501 and $3000',
            '$$3,001-$4,500': 'revenue estimates is between $3001 and $4500',
            'Greater than $4500': 'revenue estimates is above $4500'
        },
        "Grants from LWVUS": {
            '$500 or less': 'revenue estimates is less or equal to $500',
            '$501-$1,500': 'revenue estimates is between $501 to $1500',
            '$1,501-$3,000': 'revenue estimates is between $1501 and $3000',
            '$$3,001-$4,500': 'revenue estimates is between $3001 and $4500',
            'Greater than $4500': 'revenue estimates is above $4500'
        },
        "External grants": {
            '$500 or less': 'revenue estimates is less or equal to $500',
            '$501-$1,500': 'revenue estimates is between $501 to $1500',
            '$1,501-$3,000': 'revenue estimates is between $1501 and $3000',
            '$$3,001-$4,500': 'revenue estimates is between $3001 and $4500',
              'Greater than $4500': 'revenue estimates is above $4500'
              },
        "Fundaraising efforts": {
            '$500 or less': 'revenue estimates is less or equal to $500',
            '$501-$1,500': 'revenue estimates is between $501 to $1500',
            '$1,501-$3,000': 'revenue estimates is between $1501 and $3000',
            '$$3,001-$4,500': 'revenue estimates is between $3001 and $4500',
            'Greater than $4500': 'revenue estimates is above $4500'
        },
        "Individual donations from your league members": {
            '$500 or less': 'revenue estimates is less or equal to $500',
            '$501-$1,500': 'revenue estimates is between $501 to $1500',
            '$1,501-$3,000': 'revenue estimates is between $1501 and $3000',
            '$$3,001-$4,500': 'revenue estimates is between $3001 and $4500',
            'Greater than $4500': 'revenue estimates is above $4500'
        },
        "In kind donations(not including volunteer time)": {
            '$500 or less': 'revenue estimates is less or equal to $500',
            '$501-$1,500': 'revenue estimates is between $501 to $1500',
            '$1,501-$3,000': 'revenue estimates is between $1501 and $3000',
            '$$3,001-$4,500': 'revenue estimates is between $3001 and $4500',
            'Greater than $4500': 'revenue estimates is above $4500'
        },
    }
}

def standardize_revenue_estimates(row):
    results = []
    for col in revenue_estimates_column:
        if pd.notnull(row[col]):
            source = col.split('_')[1].capitalize()  # e.g. 'Savings'
            amount = row[col]
            mapped_text = f"{mappings[56].get(source, source)} is {amount}"
            results.append(mapped_text)
    return '; '.join(results)

table_df['Revenue Source_Standardized'] = table_df.apply(standardize_revenue_estimates, axis=1)

In [ ]:
# 58

league_dedicated_position_columns = [
    'fundraising_58_does_your_league_have_any_dedicated_positions_paid_or_volunteer_for_the_following_roles',
    'administrative_support_58_does_your_league_have_any_dedicated_positions_paid_or_volunteer_for_the_following_roles',
    'communicationsmarketing_58_does_your_league_have_any_dedicated_positions_paid_or_volunteer_for_the_following_roles',
    'membership_coordination_58_does_your_league_have_any_dedicated_positions_paid_or_volunteer_for_the_following_roles',
    'advocacyprogram_coordination_58_does_your_league_have_any_dedicated_positions_paid_or_volunteer_for_the_following_roles'
] 

column_map = {
    58: {
        'Fundraising': {
            'le': 'option of le',
            'Volunteer': 'volunteering',
            'No/N/A': 'there"s none'
        },
        'Administrative Support': {
            'le': 'option of le',
            'Volunteer': 'volunteering',
            'No/N/A': 'there"s none'
        },
        'Communications/Marketing': {
            'le': 'option of le',
            'Volunteer': 'volunteering',
            'No/N/A': "there's none"
        },
        'Membership Coordination': {
            'le': 'option of le',
            'Volunteer': 'volunteering',
            'No/N/A': 'there"s none'
        },
        'Advocacy/Program Coordination': {
            'le': 'option of le',
            'Volunteer': 'volunteering',
            'No/N/A': 'there"s none'
        }
    },
}

def standardize_league_dedicated_positions(row):
    results = []
    for col in league_dedicated_position_columns:
        value = row[col]
        if pd.notnull(value):
            mapped = column_map[58].get(value, value)
            results.append(f"{col}: {mapped}")
    return '; '.join(results)

table_df['League_Dedicated_Positions_Standardized'] = table_df.apply(standardize_league_dedicated_positions, axis=1)

In [ ]:
# 62

mapping_dict = {
    62: {
        'Yes': 'league was equipped',
        'No': "league wasn't equipped"
    }
}

table_df['League_Equipped_Standardized'] = table_df['62_did_your_league_feel_well_equipped_for_the_november_2024_elections'].str.strip().str.capitalize().map(mapping_dict[62])

In [ ]:
table_df.to_csv('cleaned_survey_data.csv', index=False)